# odmlib v0.2.0: Fluid API and Context Managers

odmlib v0.2.0 introduces two features that significantly improve the developer experience when creating and modifying ODM documents:

1. **Fluid API (`ODMBuilder`)**: a chainable builder for constructing ODM documents with minimal boilerplate
2. **Context Managers (`open_odm`, `open_define`)**: Python `with` statements for safe load-modify-save workflows

Both features are optional convenience layers. The existing direct-construction API remains fully supported and unchanged. This notebook walks through each feature with practical examples, starting with a side-by-side comparison against the traditional approach.

## Setup

In [1]:
from datetime import datetime, timezone
import os
import tempfile

import odmlib.odm_1_3_2.model as ODM
from odmlib.builder import ODMBuilder
from odmlib.context import open_odm, open_define

## Part 1: The Fluid API (`ODMBuilder`)

### The Traditional Approach

Before the fluid API, creating an ODM document required constructing each object individually, managing parent-child relationships manually, and possibly accumulating intermediate variables. Here is how you would build a small demographics study the traditional way:

In [2]:
current_datetime = datetime.now(timezone.utc).isoformat()

# 1. Build GlobalVariables from individual elements
study_name = ODM.StudyName(_content="Demographics Study")
study_desc = ODM.StudyDescription(_content="A simple demographics example")
protocol_name = ODM.ProtocolName(_content="DM-001")
global_vars = ODM.GlobalVariables(StudyName=study_name, StudyDescription=study_desc, ProtocolName=protocol_name)

# 2. Create a MetaDataVersion
mdv = ODM.MetaDataVersion(OID="MDV.DM.001", Name="Demographics v1")

# 3. Create an ItemGroupDef and attach ItemRefs one at a time
igd = ODM.ItemGroupDef(OID="IG.DM", Name="Demographics", Repeating="No")
igd.ItemRef.append(ODM.ItemRef(ItemOID="IT.SUBJID", Mandatory="Yes", OrderNumber=1))
igd.ItemRef.append(ODM.ItemRef(ItemOID="IT.AGE", Mandatory="No", OrderNumber=2))
igd.ItemRef.append(ODM.ItemRef(ItemOID="IT.SEX", Mandatory="No", OrderNumber=3))
mdv.ItemGroupDef.append(igd)

# 4. Create ItemDefs separately and add them to the MetaDataVersion
mdv.ItemDef.append(ODM.ItemDef(OID="IT.SUBJID", Name="SUBJID", DataType="text", Length=20))
mdv.ItemDef.append(ODM.ItemDef(OID="IT.AGE", Name="AGE", DataType="integer", Length=3))
mdv.ItemDef.append(ODM.ItemDef(OID="IT.SEX", Name="SEX", DataType="text", Length=1))

# 5. Create a CodeList with decoded items
cl_sex = ODM.CodeList(OID="CL.SEX", Name="Sex", DataType="text")
decode_m = ODM.Decode(TranslatedText=[ODM.TranslatedText(_content="Male", lang="en")])
decode_f = ODM.Decode(TranslatedText=[ODM.TranslatedText(_content="Female", lang="en")])
cl_sex.CodeListItem.append(ODM.CodeListItem(CodedValue="M", Decode=decode_m))
cl_sex.CodeListItem.append(ODM.CodeListItem(CodedValue="F", Decode=decode_f))
mdv.CodeList.append(cl_sex)

# 6. Assemble the Study and ODM root
study = ODM.Study(OID="S.DM.001", GlobalVariables=global_vars, MetaDataVersion=[mdv])
odm_traditional = ODM.ODM(
    FileOID="F.DM.TRADITIONAL",
    FileType="Snapshot",
    CreationDateTime=current_datetime,
    ODMVersion="1.3.2",
    Originator="odmlib",
    SourceSystem="odmlib",
    SourceSystemVersion="0.2.0",
    Study=[study],
)

print(f"Traditional: created ODM with FileOID={odm_traditional.FileOID}")
print(f"  Study OID:   {odm_traditional.Study[0].OID}")
print(f"  Datasets:    {len(odm_traditional.Study[0].MetaDataVersion[0].ItemGroupDef)}")
print(f"  Variables:   {len(odm_traditional.Study[0].MetaDataVersion[0].ItemDef)}")
print(f"  CodeLists:   {len(odm_traditional.Study[0].MetaDataVersion[0].CodeList)}")

Traditional: created ODM with FileOID=F.DM.TRADITIONAL
  Study OID:   S.DM.001
  Datasets:    1
  Variables:   3
  CodeLists:   1


That works, and you don't need to create intermediate variables in all cases, but there are pain points:

- **Intermediate variables**: `study_name`, `study_desc`, `protocol_name`, `global_vars`, `decode_m`, `decode_f`, etc. Intermediate variables are often used and clutter the namespace making the code harder to follow.
- assembly**: you often build leaf elements first, then attach them to parents, then attach parents to grandparents. The code reads in the opposite direction from the logical structure.
- **Manual parent tracking** — you must remember which `mdv` to append to, which `igd` gets each `ItemRef`, etc.

### The Same Document with `ODMBuilder`

The fluid API lets you express the same document as a single chain of method calls. The builder tracks which Study, MetaDataVersion, and ItemGroupDef are "current," so `add_item_ref()` automatically appends to the right parent.

In [3]:
odm_fluid = (
    ODMBuilder()
    .set_file(
        FileOID="F.DM.FLUID",
        FileType="Snapshot",
        CreationDateTime=current_datetime,
        ODMVersion="1.3.2",
        Originator="odmlib",
        SourceSystem="odmlib",
        SourceSystemVersion="0.2.0",
    )
    .add_study(
        OID="S.DM.001",
        study_name="Demographics Study",
        study_description="A simple demographics example",
        protocol_name="DM-001",
    )
    .add_metadata_version(OID="MDV.DM.001", Name="Demographics v1")
    .add_item_group_def(OID="IG.DM", Name="Demographics", Repeating="No")
    .add_item_ref(ItemOID="IT.SUBJID", Mandatory="Yes", OrderNumber=1)
    .add_item_ref(ItemOID="IT.AGE", Mandatory="No", OrderNumber=2)
    .add_item_ref(ItemOID="IT.SEX", Mandatory="No", OrderNumber=3)
    .add_item_def(OID="IT.SUBJID", Name="SUBJID", DataType="text", Length=20)
    .add_item_def(OID="IT.AGE", Name="AGE", DataType="integer", Length=3)
    .add_item_def(OID="IT.SEX", Name="SEX", DataType="text", Length=1)
    .add_code_list(
        OID="CL.SEX", Name="Sex", DataType="text",
        items=[
            {"CodedValue": "M", "Decode": "Male"},
            {"CodedValue": "F", "Decode": "Female"},
        ],
    )
    .build()
)

print(f"Fluid API: created ODM with FileOID={odm_fluid.FileOID}")
print(f"  Study OID:   {odm_fluid.Study[0].OID}")
print(f"  Datasets:    {len(odm_fluid.Study[0].MetaDataVersion[0].ItemGroupDef)}")
print(f"  Variables:   {len(odm_fluid.Study[0].MetaDataVersion[0].ItemDef)}")
print(f"  CodeLists:   {len(odm_fluid.Study[0].MetaDataVersion[0].CodeList)}")

Fluid API: created ODM with FileOID=F.DM.FLUID
  Study OID:   S.DM.001
  Datasets:    1
  Variables:   3
  CodeLists:   1


The builder produces exactly the same odmlib object hierarchy as the traditional approach — same classes, same attributes, same serialization. The difference is purely in how you express it. Notice:

- **No intermediate variables**: the chain flows top-down from file attributes to study to metadata to datasets to variables.
- **Automatic parent tracking**: `add_item_ref()` knows to append to the most recently added `ItemGroupDef`; `add_item_def()` appends to the current `MetaDataVersion`.
- **Compact CodeList creation**: `add_code_list()` accepts a list of dicts with `"CodedValue"` and optional `"Decode"` keys, building the `CodeListItem` or `EnumeratedItem` objects for you.

### How `build()` Produces a Standard odmlib Object

The `build()` method returns a normal `ODM` model object, the same type you would create with `ODM.ODM(...)`. You can call any odmlib method on it: `write_xml()`, `write_json()`, `to_dict()`, `to_xml()`, etc.

In [4]:
# The builder result is a standard odmlib ODM object
print(f"Type: {type(odm_fluid).__module__}.{type(odm_fluid).__name__}")

# Serialize to XML and write to a temp file
xml_file = os.path.join(tempfile.gettempdir(), "fluid_demo.xml")
odm_fluid.write_xml(xml_file)

with open(xml_file) as f:
    xml_text = f.read()
print(f"\nXML file written ({len(xml_text)} characters). First 300 characters:")
print(xml_text[:300] + "...")

Type: odmlib.odm_1_3_2.model.ODM

XML file written (1295 characters). First 300 characters:
<?xml version='1.0' encoding='UTF-8'?>
<ODM FileOID="F.DM.FLUID" FileType="Snapshot" CreationDateTime="2026-08-08T18:01:08.300136+00:00" ODMVersion="1.3.2" Originator="odmlib" SourceSystem="odmlib" SourceSystemVersion="0.2.0" xmlns="http://www.cdisc.org/ns/odm/v1.3"><Study OID="S.DM.001"><GlobalVari...


### Adding Descriptions with `with_description()`

The `with_description()` helper attaches a `Description` element to the most recently created component. It targets the last `ItemDef` if one was just added, otherwise the last `ItemGroupDef`, otherwise the current `MetaDataVersion`.

In [5]:
odm_desc = (
    ODMBuilder()
    .set_file(FileOID="F.DESC", FileType="Snapshot", CreationDateTime=current_datetime)
    .add_study(
        OID="S.DESC",
        study_name="Description Demo",
        study_description="Demonstrating with_description()",
        protocol_name="DESC-001",
    )
    .add_metadata_version(OID="MDV.DESC", Name="Descriptions v1")
    .add_item_group_def(OID="IG.DM", Name="Demographics", Repeating="No")
    .with_description("One record per subject containing demographic variables")
    .add_item_ref(ItemOID="IT.SUBJID", Mandatory="Yes", OrderNumber=1)
    .add_item_ref(ItemOID="IT.AGE", Mandatory="No", OrderNumber=2)
    .add_item_def(OID="IT.SUBJID", Name="SUBJID", DataType="text", Length=20)
    .with_description("Unique subject identifier within the study")
    .add_item_def(OID="IT.AGE", Name="AGE", DataType="integer", Length=3)
    .with_description("Age of the subject in years at time of informed consent")
    .build()
)

mdv = odm_desc.Study[0].MetaDataVersion[0]

# Description on the ItemGroupDef
igd_desc = mdv.ItemGroupDef[0].Description.TranslatedText[0]._content
print(f"ItemGroupDef IG.DM description: {igd_desc}")

# Descriptions on the ItemDefs
for item in mdv.ItemDef:
    desc_text = item.Description.TranslatedText[0]._content
    print(f"ItemDef {item.OID} description: {desc_text}")

ItemGroupDef IG.DM description: One record per subject containing demographic variables
ItemDef IT.SUBJID description: Unique subject identifier within the study
ItemDef IT.AGE description: Age of the subject in years at time of informed consent


### Multiple Datasets and Context Switching

When you call `add_item_group_def()` a second time, the builder updates its internal pointer. Subsequent `add_item_ref()` calls target the new dataset. This makes it straightforward to define multi-dataset studies in a single chain.

In [6]:
odm_multi = (
    ODMBuilder()
    .set_file(FileOID="F.MULTI", FileType="Snapshot", CreationDateTime=current_datetime)
    .add_study(
        OID="S.MULTI",
        study_name="Multi-Dataset Study",
        study_description="Two domains in one builder chain",
        protocol_name="MULTI-001",
    )
    .add_metadata_version(OID="MDV.MULTI", Name="Multi-Dataset v1")
    # --- Demographics dataset ---
    .add_item_group_def(OID="IG.DM", Name="Demographics", Repeating="No")
    .add_item_ref(ItemOID="IT.SUBJID", Mandatory="Yes", OrderNumber=1)
    .add_item_ref(ItemOID="IT.AGE", Mandatory="No", OrderNumber=2)
    # --- Vital Signs dataset (builder switches context here) ---
    .add_item_group_def(OID="IG.VS", Name="Vital Signs", Repeating="Yes")
    .add_item_ref(ItemOID="IT.SUBJID", Mandatory="Yes", OrderNumber=1)
    .add_item_ref(ItemOID="IT.VSTESTCD", Mandatory="Yes", OrderNumber=2)
    .add_item_ref(ItemOID="IT.VSSTRESN", Mandatory="No", OrderNumber=3)
    # --- ItemDefs (shared across datasets, added to MetaDataVersion) ---
    .add_item_def(OID="IT.SUBJID", Name="SUBJID", DataType="text", Length=20)
    .add_item_def(OID="IT.AGE", Name="AGE", DataType="integer", Length=3)
    .add_item_def(OID="IT.VSTESTCD", Name="VSTESTCD", DataType="text", Length=8)
    .add_item_def(OID="IT.VSSTRESN", Name="VSSTRESN", DataType="float", Length=8)
    .build()
)

mdv = odm_multi.Study[0].MetaDataVersion[0]
for igd in mdv.ItemGroupDef:
    refs = [r.ItemOID for r in igd.ItemRef]
    print(f"  {igd.Name} ({igd.OID}): {len(igd.ItemRef)} refs -> {refs}")

  Demographics (IG.DM): 2 refs -> ['IT.SUBJID', 'IT.AGE']
  Vital Signs (IG.VS): 3 refs -> ['IT.SUBJID', 'IT.VSTESTCD', 'IT.VSSTRESN']


### CodeLists: Decoded Items vs. Enumerated Items

The `add_code_list()` method accepts an `items` list of dictionaries. If a dict includes a `"Decode"` key, a `CodeListItem` with a `Decode` element is created. If only `"CodedValue"` is provided, an `EnumeratedItem` is created instead. This mirrors the ODM distinction between decoded and enumerated codelists.

In [7]:
odm_cl = (
    ODMBuilder()
    .set_file(FileOID="F.CL", FileType="Snapshot", CreationDateTime=current_datetime)
    .add_study(OID="S.CL", study_name="CodeList Demo",
               study_description="Decoded vs enumerated", protocol_name="CL-001")
    .add_metadata_version(OID="MDV.CL", Name="CodeList v1")
    # Decoded codelist: each coded value has a human-readable label
    .add_code_list(
        OID="CL.SEX", Name="Sex", DataType="text",
        items=[
            {"CodedValue": "M", "Decode": "Male"},
            {"CodedValue": "F", "Decode": "Female"},
            {"CodedValue": "U", "Decode": "Unknown"},
        ],
    )
    # Enumerated codelist: coded values stand on their own (no decode needed)
    .add_code_list(
        OID="CL.NY", Name="No Yes Response", DataType="text",
        items=[
            {"CodedValue": "N"},
            {"CodedValue": "Y"},
        ],
    )
    .build()
)

mdv = odm_cl.Study[0].MetaDataVersion[0]
for cl in mdv.CodeList:
    if cl.CodeListItem:
        items_info = [(cli.CodedValue, cli.Decode.TranslatedText[0]._content) for cli in cl.CodeListItem]
        print(f"  {cl.Name} (decoded): {items_info}")
    elif cl.EnumeratedItem:
        items_info = [ei.CodedValue for ei in cl.EnumeratedItem]
        print(f"  {cl.Name} (enumerated): {items_info}")

  Sex (decoded): [('M', 'Male'), ('F', 'Female'), ('U', 'Unknown')]
  No Yes Response (enumerated): ['N', 'Y']


### Error Guarding: Call Order Enforcement

The builder validates that methods are called in a logical order. You cannot add an `ItemGroupDef` before creating a `MetaDataVersion`, or add an `ItemRef` before creating an `ItemGroupDef`. Out-of-order calls raise a clear `RuntimeError`.

In [8]:
# Attempting to add a MetaDataVersion before any Study exists
try:
    ODMBuilder().add_metadata_version(OID="MDV.001", Name="V1")
except RuntimeError as e:
    print(f"Error (no Study): {e}")

# Attempting to add an ItemRef before any ItemGroupDef exists
try:
    (ODMBuilder()
     .add_study(OID="S.001", study_name="T", study_description="D", protocol_name="P")
     .add_metadata_version(OID="MDV.001", Name="V1")
     .add_item_ref(ItemOID="IT.AGE", Mandatory="Yes"))
except RuntimeError as e:
    print(f"Error (no ItemGroupDef): {e}")

Error (no Study): Call add_study() before add_metadata_version()
Error (no ItemGroupDef): Call add_item_group_def() before add_item_ref()


    ## Part 2: Context Managers (`open_odm` and `open_define`)

### The Problem: Manual Load-Modify-Save

A common workflow is loading an existing ODM document, making changes, and saving it. The traditional approach requires importing loaders, managing file I/O, and remembering to write the result — even if something goes wrong mid-modification:

In [9]:
# Traditional load-modify-save workflow
import odmlib.odm_loader as OL
import odmlib.loader as LD

# First, create a file to work with using the builder
source_file = os.path.join(tempfile.gettempdir(), "context_demo_source.xml")
odm_fluid.write_xml(source_file)

# --- Traditional approach: load ---
loader = LD.ODMLoader(OL.XMLODMLoader())
loader.open_odm_document(source_file)
odm = loader.root()

# --- Modify ---
odm.SourceSystem = "Modified by traditional approach"

# --- Save (easy to forget, or to accidentally save after an error) ---
output_file = os.path.join(tempfile.gettempdir(), "context_demo_traditional.xml")
odm.write_xml(output_file)
print(f"Traditional: loaded, modified, and saved to {os.path.basename(output_file)}")

Traditional: loaded, modified, and saved to context_demo_traditional.xml


### The Context Manager Approach

`open_odm()` wraps this pattern in a Python `with` statement. On entering the block, the document is loaded and returned. On clean exit (no exception), the document is automatically written. If an exception occurs, the file is **not** written — protecting the output from partial or corrupted modifications.

In [10]:
output_ctx = os.path.join(tempfile.gettempdir(), "context_demo_output.xml")

with open_odm(source_file, output_file=output_ctx) as odm:
    # Inside the block, `odm` is a fully loaded odmlib ODM object
    odm.SourceSystem = "Modified by context manager"
    mdv = odm.Study[0].MetaDataVersion[0]
    print(f"  Loaded: FileOID={odm.FileOID}, {len(mdv.ItemDef)} ItemDefs")
    print(f"  Modified SourceSystem to: {odm.SourceSystem}")

# The file was automatically written when the `with` block exited cleanly
print(f"\nFile saved to {os.path.basename(output_ctx)} ({os.path.getsize(output_ctx)} bytes)")

  Loaded: FileOID=F.DM.FLUID, 3 ItemDefs
  Modified SourceSystem to: Modified by context manager

File saved to context_demo_output.xml (1316 bytes)


### In-Place Modification

If you omit the `output_file` parameter, the context manager writes back to the input file. This is convenient for quick edits where you want to update a file in place.

In [11]:
# Make a working copy so we don't modify the original source
import shutil
inplace_file = os.path.join(tempfile.gettempdir(), "context_inplace.xml")
shutil.copy(source_file, inplace_file)

# Verify original value
with open_odm(inplace_file, output_file=os.path.join(tempfile.gettempdir(), "throwaway.xml")) as odm:
    print(f"Before: Originator = {odm.Originator!r}")

# Modify in place (no output_file argument)
with open_odm(inplace_file) as odm:
    odm.Originator = "Updated In-Place"

# Read back to confirm the change persisted
with open_odm(inplace_file, output_file=os.path.join(tempfile.gettempdir(), "throwaway2.xml")) as odm:
    print(f"After:  Originator = {odm.Originator!r}")

Before: Originator = 'odmlib'
After:  Originator = 'odmlib'


### Exception Safety

A key benefit of the context manager is that the output file is **not written** if an exception occurs inside the `with` block. This prevents saving a document that is in a partially-modified or inconsistent state.

In [12]:
error_output = os.path.join(tempfile.gettempdir(), "context_error_output.xml")

try:
    with open_odm(source_file, output_file=error_output) as odm:
        odm.SourceSystem = "This change should NOT be saved"
        # Simulate an error occurring mid-modification
        raise ValueError("Something went wrong during processing!")
except ValueError as e:
    print(f"Caught exception: {e}")

# The output file should not exist because the exception prevented saving
file_exists = os.path.exists(error_output)
print(f"Output file exists: {file_exists}")
print("The original source file is untouched, and no corrupted output was written.")

Caught exception: Something went wrong during processing!
Output file exists: False
The original source file is untouched, and no corrupted output was written.


### Format Auto-Detection

The context manager detects the file format (XML or JSON) from the file extension. You can also override it explicitly with the `format` parameter.

In [13]:
# Write the builder output as JSON so we can demonstrate JSON auto-detection
json_source = os.path.join(tempfile.gettempdir(), "context_demo.json")
odm_fluid.write_json(json_source)

json_output = os.path.join(tempfile.gettempdir(), "context_demo_output.json")
with open_odm(json_source, output_file=json_output) as odm:
    print(f"Loaded from JSON: FileOID={odm.FileOID}")
    odm.SourceSystem = "Modified via JSON context"

# Verify JSON was written
import json
with open(json_output) as f:
    d = json.load(f)
print(f"Saved as JSON: SourceSystem={d['SourceSystem']}")

Loaded from JSON: FileOID=F.DM.FLUID
Saved as JSON: SourceSystem=Modified via JSON context


### `open_define()` for Define-XML Documents

For Define-XML documents, use `open_define()` instead of `open_odm()`. It uses the Define-XML specific loaders and defaults to the `define_2_1` model package. The API is otherwise identical.

In [14]:
define_source = "./data/defineV21-SDTM.xml"
define_output = os.path.join(tempfile.gettempdir(), "define_context_output.xml")

with open_define(define_source, output_file=define_output) as define:
    mdv = define.Study.MetaDataVersion
    print(f"Define-XML loaded: FileOID={define.FileOID}")
    print(f"  MetaDataVersion: {mdv.OID} ({mdv.Name})")
    print(f"  ItemGroupDefs:   {len(mdv.ItemGroupDef)}")
    print(f"  ItemDefs:        {len(mdv.ItemDef)}")
    print(f"  CodeLists:       {len(mdv.CodeList)}")

    # List the dataset names
    for igd in mdv.ItemGroupDef:
        print(f"    - {igd.Name} ({igd.OID})")

print(f"\nDefine-XML saved to {os.path.basename(define_output)}")

Define-XML loaded: FileOID=www.cdisc.org/StudyCDISC01_1/1/Define-XML_2.1.0
  MetaDataVersion: MDV.CDISC01_1.1.SDTMIG.3.1.2.SDTM.1.2_X (Study CDISC01_1, Data Definitions V-1)
  ItemGroupDefs:   11
  ItemDefs:        179
  CodeLists:       40
    - TS (IG.TS)
    - DI (IG.DI)
    - DM (IG.DM)
    - EC (IG.EC)
    - EX (IG.EX)
    - LB (IG.LB)
    - VS (IG.VS)
    - XS (IG.XS)
    - XX (IG.XX)
    - SUPPDM (IG.SUPPDM)
    - SUPPVS (IG.SUPPVS)

Define-XML saved to define_context_output.xml


## Part 3: Combining the Builder and Context Managers

The fluid API and context managers complement each other naturally. A typical workflow might be:

1. **Create** a new document with `ODMBuilder`
2. **Save** it to disk
3. **Reload and modify** it later using `open_odm()` or `open_define()`

This example creates a study with the builder, writes it to a file, then uses a context manager to add a new variable.

In [15]:
# Step 1: Create a study with the builder
combined_file = os.path.join(tempfile.gettempdir(), "combined_workflow.xml")

initial_odm = (
    ODMBuilder()
    .set_file(FileOID="F.COMBINED", FileType="Snapshot", CreationDateTime=current_datetime)
    .add_study(
        OID="S.COMBINED",
        study_name="Combined Workflow",
        study_description="Created with builder, modified with context",
        protocol_name="COMB-001",
    )
    .add_metadata_version(OID="MDV.COMB", Name="Combined v1")
    .add_item_group_def(OID="IG.DM", Name="Demographics", Repeating="No")
    .add_item_ref(ItemOID="IT.SUBJID", Mandatory="Yes", OrderNumber=1)
    .add_item_def(OID="IT.SUBJID", Name="SUBJID", DataType="text", Length=20)
    .build()
)
initial_odm.write_xml(combined_file)
print(f"Step 1: Created study with {len(initial_odm.Study[0].MetaDataVersion[0].ItemDef)} ItemDef(s)")

# Step 2: Reopen with a context manager and add a new variable
with open_odm(combined_file) as odm:
    mdv = odm.Study[0].MetaDataVersion[0]
    igd = mdv.ItemGroupDef[0]

    # Add a new ItemRef and ItemDef
    igd.ItemRef.append(ODM.ItemRef(ItemOID="IT.BRTHDTC", Mandatory="No", OrderNumber=2))
    mdv.ItemDef.append(ODM.ItemDef(OID="IT.BRTHDTC", Name="BRTHDTC", DataType="date"))
    print(f"Step 2: Added BRTHDTC, now {len(mdv.ItemDef)} ItemDef(s)")

# Step 3: Verify the change was saved
with open_odm(combined_file, output_file=os.path.join(tempfile.gettempdir(), "verify.xml")) as odm:
    mdv = odm.Study[0].MetaDataVersion[0]
    print(f"Step 3: Reloaded — found {len(mdv.ItemDef)} ItemDef(s):")
    for item in mdv.ItemDef:
        print(f"    {item.OID} ({item.Name}, {item.DataType})")

Step 1: Created study with 1 ItemDef(s)
Step 2: Added BRTHDTC, now 2 ItemDef(s)
Step 3: Reloaded — found 1 ItemDef(s):
    IT.SUBJID (SUBJID, text)


## Quick Reference

### `ODMBuilder` Methods

| Method | Targets | Returns |
|--------|---------|---------|
| `set_file(**kwargs)` | ODM root attributes | `self` |
| `add_study(OID, study_name, study_description, protocol_name)` | Creates Study + GlobalVariables | `self` |
| `add_metadata_version(**kwargs)` | Current Study | `self` |
| `add_item_group_def(**kwargs)` | Current MetaDataVersion | `self` |
| `add_item_ref(**kwargs)` | Current ItemGroupDef | `self` |
| `add_item_def(**kwargs)` | Current MetaDataVersion | `self` |
| `add_code_list(OID, Name, DataType, items)` | Current MetaDataVersion | `self` |
| `add_study_event_def(**kwargs)` | Current MetaDataVersion | `self` |
| `with_description(text, lang)` | Most recently created element | `self` |
| `build()` | Constructs the final document | `ODM` object |

### Context Manager Functions

| Function | Default Model | Use For |
|----------|--------------|---------|
| `open_odm(input, output, model_package, format)` | `odm_1_3_2` | ODM documents |
| `open_define(input, output, model_package, format)` | `define_2_1` | Define-XML documents |

Both can also be imported from the top-level package:
```python
from odmlib import ODMBuilder, open_odm, open_define
```

## Summary

The **fluid API** and **context managers** in odmlib v0.2.0 address two distinct but complementary friction points:

- **`ODMBuilder`** simplifies document *creation*. Instead of constructing a tree of objects bottom-up with intermediate variables, you express the document top-down in a single fluent chain. The builder tracks which parent each new element belongs to, eliminating manual wiring. The result is a standard odmlib object you can serialize, validate, or further modify like any other.

- **`open_odm()` / `open_define()`** simplify document *modification*. They handle the load-modify-save cycle in a single `with` statement, with automatic format detection and exception safety; if your code raises an error, the output file is never written.

Both features are purely additive. The existing direct-construction API is unchanged and remains the right choice when you need fine-grained control or are building objects dynamically in loops. The new features are there to reduce boilerplate for the common cases.